## RCA Data Transformations

This notebook uses `transform_incidents.py` to build two derived datasets from `data/synthetic_source.jsonl`:

- **Agent 1 (structured RCA)**: flattened + feature-engineered tabular view for tree-based models.
- **Agent 2 (semantic / evidence)**: combined evidence and target text for retrieval / LLM / RAG pipelines.

In [1]:
import pandas as pd
from pathlib import Path

from transform_incidents import (
    load_incidents,
    build_agent1_structured,
    build_agent2_evidence,
)

RAW_PATH = Path("data/synthetic_source.jsonl")
OUTDIR = Path("data/processed")
OUTDIR.mkdir(parents=True, exist_ok=True)

RAW_PATH, OUTDIR

(PosixPath('data/synthetic_source.jsonl'), PosixPath('data/processed'))

### Load raw incidents

Each row is one incident with nested JSON fields.

In [2]:
df_raw = load_incidents(RAW_PATH)
print(f"Loaded {len(df_raw):,} raw incidents")
df_raw.head(2)

Loaded 8,502 raw incidents


,id,context,world,fault,observations,remediation,meta
0,6d40f737-8acb-432b-a235-5bf301b92326,"{'cluster_id': 'cust-166', 'namespace': 'team-...","{'nodes': [{'name': 'node-zdoc9i', 'allocatabl...",{'scenario_id': 'createcontainerconfigerror_mi...,{'kubectl_get_pods': 'NAME READY STATUS RESTAR...,"{'category': 'CreateContainerConfigError', 'di...","{'created_at': '2026-03-18T04:27:46Z', 'diffic..."
1,1d9505e7-19b0-49ae-9b70-94275691375e,"{'cluster_id': 'cust-336', 'namespace': 'payme...","{'nodes': [{'name': 'node-gxbeny', 'allocatabl...",{'scenario_id': 'createcontainerconfigerror_mi...,{'kubectl_get_pods': 'NAME READY STATUS RESTAR...,"{'category': 'CreateContainerConfigError', 'di...","{'created_at': '2026-03-18T04:27:46Z', 'diffic..."


### Build Agent 1 (structured RCA) dataset

Flatten nested fields, parse K8s observations, and derive structured RCA features.

In [3]:
agent1_df = build_agent1_structured(df_raw)
print(f"Agent 1 structured rows: {len(agent1_df):,}")
agent1_df.head()

Agent 1 structured rows: 8,502


,id,cluster_id,namespace,workload_kind,workload_name,container_name,image,scenario_id,variant,category,...,restart_count,event_type,event_reason,event_message,restart_count_metrics,oom_killed,missing_secret,missing_key,symptom_family,root_cause_family
0,6d40f737-8acb-432b-a235-5bf301b92326,cust-166,team-a-dev,Deployment,module-w-ncf1,module-w-ncf1-c,ghcr.io/acme/backend:1.20.20,createcontainerconfigerror_missing_secret,secret_not_found,CreateContainerConfigError,...,0,Warning,Failed,"Error: secret ""api-secrets"" not found",None,None,None,None,CreateContainerConfigError,missing_secret
1,1d9505e7-19b0-49ae-9b70-94275691375e,cust-336,payments-prod,Deployment,backend-w-dmen,backend-w-dmen-c,ghcr.io/acme/app:latest,createcontainerconfigerror_missing_secret,secret_not_found,CreateContainerConfigError,...,0,Warning,Failed,"Error: secret ""db-credentials"" not found",None,None,None,None,CreateContainerConfigError,missing_secret
2,e03ec390-f797-4991-8b2d-1a3ddd2afbb8,cust-167,payments-dev,Deployment,service-w-g629,service-w-g629-c,ghcr.io/acme/worker:5.8.14,createcontainerconfigerror_missing_secret,secret_not_found,CreateContainerConfigError,...,0,Warning,Failed,"Error: secret ""s3-keys"" not found",None,None,None,None,CreateContainerConfigError,missing_secret
3,2604b682-3d75-43d2-b467-702f6138da37,cust-169,team-b-stg,Deployment,app-w-f4r6,main,docker.io/acme/app:4.1.12,createcontainerconfigerror_missing_secret,secret_not_found,CreateContainerConfigError,...,0,Warning,Failed,"Error: secret ""payments-db"" not found",None,None,None,None,CreateContainerConfigError,missing_secret
4,f095d06e-e983-4379-8f4c-4f0064f55387,cust-62,team-a-prod,Deployment,module-w-ex1r,main,ghcr.io/acme/worker:2.13.16,createcontainerconfigerror_missing_secret,secret_not_found,CreateContainerConfigError,...,0,Warning,Failed,"Error: secret ""payments-db"" not found",None,None,None,None,CreateContainerConfigError,missing_secret


### Build Agent 2 (semantic / evidence) dataset

Combine evidence text and remediation text targets for text-based RCA models.

In [4]:
agent2_df = build_agent2_evidence(df_raw)
print(f"Agent 2 evidence rows: {len(agent2_df):,}")
agent2_df[["id", "scenario_id", "namespace", "difficulty", "noise_level"]].head()

Agent 2 evidence rows: 8,502


,id,scenario_id,namespace,difficulty,noise_level
0,6d40f737-8acb-432b-a235-5bf301b92326,createcontainerconfigerror_missing_secret,team-a-dev,medium,2
1,1d9505e7-19b0-49ae-9b70-94275691375e,createcontainerconfigerror_missing_secret,payments-prod,medium,1
2,e03ec390-f797-4991-8b2d-1a3ddd2afbb8,createcontainerconfigerror_missing_secret,payments-dev,easy,1
3,2604b682-3d75-43d2-b467-702f6138da37,createcontainerconfigerror_missing_secret,team-b-stg,medium,2
4,f095d06e-e983-4379-8f4c-4f0064f55387,createcontainerconfigerror_missing_secret,team-a-prod,easy,0


### Write transformed datasets to Parquet

This mirrors the CLI (`python transform_incidents.py --input data/synthetic_source.jsonl --outdir data/processed/`).

In [5]:
agent1_path = OUTDIR / "agent1_structured.parquet"
agent2_path = OUTDIR / "agent2_evidence.parquet"

agent1_df.to_parquet(agent1_path, index=False)
agent2_df.to_parquet(agent2_path, index=False)

agent1_path, agent2_path

(PosixPath('data/processed/agent1_structured.parquet'),
 PosixPath('data/processed/agent2_evidence.parquet'))